In [ ]:
!pip install sqlalchemy

In [ ]:
!pip install pymysql

In [ ]:
pip show sqlalchemy | grep Version

In [ ]:
pip install --upgrade sqlalchemy

In [1]:
import pandas as pd
from sqlalchemy import create_engine
import getpass

password = getpass.getpass()

bd = "sakila"
connection_string = 'mysql+pymysql://root:' + password + '@localhost/' + bd
engine = create_engine(connection_string)

 ········


In [2]:
def rentals_month(engine, month, year):
    query = f"""
        SELECT *
        FROM rental
        WHERE MONTH(rental_date) = {month} AND YEAR(rental_date) = {year}
    """
    df = pd.read_sql(query, engine)
    return df

In [3]:
may_rentals = rentals_month(engine, 5, 2005)
may_rentals.head()

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


In [4]:
def rental_count_month(df, month, year):
    column_name = f"rentals_{month:02d}_{year}"
    result = df.groupby('customer_id').size().reset_index(name=column_name)
    return result

In [5]:
rentals_05_2005 = rental_count_month(may_rentals, 5, 2005)
rentals_05_2005.head()

,customer_id,rentals_05_2005
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3


In [6]:
june_rentals = rentals_month(engine, 6, 2005)
rentals_06_2005 = rental_count_month(june_rentals, 6, 2005)

In [7]:
def compare_rentals(df1, df2):
    combined = pd.merge(df1, df2, on='customer_id', how='outer').fillna(0)
    col1 = df1.columns[1]
    col2 = df2.columns[1]
    combined['difference'] = combined[col2] - combined[col1]
    return combined

In [8]:
comparison = compare_rentals(rentals_05_2005, rentals_06_2005)
comparison.head()

,customer_id,rentals_05_2005,rentals_06_2005,difference
0,1,2.0,7.0,5.0
1,2,1.0,1.0,0.0
2,3,2.0,4.0,2.0
3,4,0.0,6.0,6.0
4,5,3.0,5.0,2.0
